In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 20 — LIME: EXPLICANDO MODELOS COM APROXIMAÇÕES LOCAIS

> "Se você não consegue entender um mecanismo complexo, não tente desmontá-lo inteiro. Observe como ele se comporta numa pequena vizinhança. Aproxime a complexidade com a simplicidade local."
> — Um Engenheiro de Depuração

O SHAP me deu uma contabilidade rigorosa, mas complexa. Enquanto preparava a apresentação, me perguntei se haveria uma forma mais intuitiva de explicar uma previsão — algo que eu pudesse esboçar num guardanapo para o chefe da oncologia.

A resposta foi o **LIME**. Sua filosofia é brilhante em sua simplicidade: "não me importa quão complexa é a fronteira do seu modelo no geral; quero entender o que ela faz *aqui mesmo*, na vizinhança deste paciente". Para isso, o LIME gera amostras "fantasmas" ao redor do paciente, pergunta ao modelo o que ele prevê para cada uma, e treina um modelo simples e interpretável nessa vizinhança. É como colocar uma lupa numa pequena porção da fronteira e ver que, ali, ela se parece com uma reta.

## LIME: Fidelidade Local, Simplicidade Global

O LIME explica uma previsão em cinco passos: (1) escolhe a instância; (2) gera amostras perturbadas em sua vizinhança; (3) obtém as previsões do modelo complexo para elas; (4) pondera as amostras pela proximidade; (5) treina um modelo linear simples nesse conjunto ponderado. Os coeficientes desse modelo são a explicação. O objetivo é a **fidelidade local**: imitar o modelo complexo *naquela vizinhança*, não globalmente.

| Característica | LIME | SHAP |
|---|---|---|
| Fundamento | Aproximação local com modelo simples | Teoria dos jogos (valores de Shapley) |
| Garantias | Nenhuma garantia teórica forte | Fortes (eficiência, consistência) |
| Velocidade | Geralmente rápido | KernelExplainer lento; TreeExplainer rápido |
| Intuição | Muito intuitivo (aproximar curva por reta) | Menos intuitivo (coalizões) |

## Explicando o Paciente #11 Com LIME

Vamos explicar o **mesmo paciente maligno** do capítulo anterior — do treino, corretamente classificado, com **98,7%** de probabilidade — para comparar as duas ferramentas. O LIME usa amostragem aleatória, então fixamos `random_state=42` para reprodutibilidade (uma limitação real da técnica, que discutimos adiante).

#### Código 20.1: Explicação Local com LIME


In [ ]:
import lime, lime.lime_tabular
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.svm import SVC
from oncoclassify_utils import X_train, y_train, FEATURES_MORFOLOGICAS

def info_mutua(X, y): return mutual_info_classif(X, y, random_state=42)
modelo = Pipeline([("selecao", SelectKBest(info_mutua, k=25)),
                   ("escala", StandardScaler()),
                   ("svm", SVC(kernel="rbf", C=10, gamma=0.01,
                               probability=True, random_state=42))]).fit(X_train, y_train)

explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train.values, feature_names=list(FEATURES_MORFOLOGICAS),
    class_names=["Maligno", "Benigno"], mode="classification", random_state=42)

paciente = 0   # primeiro paciente do treino (o cofre X_test segue lacrado)
exp = explainer.explain_instance(X_train.iloc[paciente].values, modelo.predict_proba,
                                 num_features=6, labels=(0,), num_samples=5000)

prob = modelo.predict_proba(X_train.iloc[[paciente]])[0]
print(f"Paciente #{paciente}: P(Maligno)={prob[0]:.3f}  P(Benigno)={prob[1]:.3f}  (real: Maligno)\n")
for regra, peso in exp.as_list(label=0):
    lado = "-> MALIGNO" if peso > 0 else "-> benigno"
    print(f"  {peso:+.4f}  {lado}   {regra}")

Paciente #0: P(Maligno)=0.987  P(Benigno)=0.013  (real: Maligno)

  +0.2257  -> MALIGNO   worst concave points > 0.16
  -0.2038  -> benigno   compactness error > 0.03
  -0.1736  -> benigno   mean compactness > 0.13
  +0.1733  -> MALIGNO   mean concave points > 0.07
  +0.1689  -> MALIGNO   worst perimeter > 124.15
  +0.1668  -> MALIGNO   worst concavity > 0.38


![Explicação LIME — paciente maligno do treino](media/figura_20_1.png)

A leitura é direta e acionável. Para este paciente, quatro regras empurram para **maligno** — a mais forte, `worst concave points > 0.16` (+0,23), seguida de `mean concave points > 0.07`, `worst perimeter > 124.15` e `worst concavity > 0.38`. Duas regras puxam para **benigno** (`compactness error > 0.03` e `mean compactness > 0.13`), mas a evidência maligna prevalece com folga — coerente com os 98,7% de probabilidade.

Repare na convergência com o SHAP do capítulo anterior: ambas as ferramentas apontam **pontos côncavos e concavidade** como os fatores decisivos deste caso. Quando duas técnicas independentes concordam, a confiança na explicação cresce.

> **⚠️ ATENÇÃO — A instabilidade do LIME**
>
> Como o LIME se baseia em amostragem aleatória na vizinhança, execuções diferentes podem produzir explicações ligeiramente distintas — por isso fixamos a semente. Essa é sua principal fraqueza frente ao SHAP, que tem garantias teóricas. Use o LIME para explicações rápidas e intuitivas; recorra ao SHAP quando precisar de rigor e reprodutibilidade forte.

> **📌 CHECKLIST DO CAPÍTULO 20**
>
> ✅ Entende a filosofia do LIME: aproximar um modelo complexo por um simples, localmente.
>
> ✅ Sabe descrever seus cinco passos e as diferenças frente ao SHAP.
>
> ✅ Executou o Código 20.1 e interpretou a explicação de um caso **maligno** real (o paciente #11).
>
> ✅ Reconhece a **instabilidade** do LIME e a necessidade de fixar a semente.
>
> **⚠️ CRÍTICO:** LIME e SHAP são complementares. A concordância entre eles é um forte sinal de que a explicação é confiável.

Minha jornada pela interpretabilidade estava completa. Da visão panorâmica (importância global) à contabilidade rigorosa (SHAP) e à aproximação intuitiva (LIME), eu não tinha mais uma caixa-preta: tinha uma caixa de vidro. Podia ver o mecanismo, auditar as decisões e traduzir sua lógica para uma linguagem humana.

Mas a confiança não vem só da explicação — vem também da consistência. Eu avaliara o modelo com validação cruzada de 5 *folds*. Seria suficiente? Como ter certeza de que meu desempenho não fora só um golpe de sorte na divisão dos dados? Eu precisava de técnicas mais rigorosas de validação. Era hora do teste de estresse final.

**PARTE VII — VALIDAÇÃO E CONFIABILIDADE**
